In [ ]:
#include <WiFi.h>
#include <WiFiUdp.h>
#include <Wire.h>
#include <TinyGPSPlus.h>
#include <Adafruit_Sensor.h>
#include <Adafruit_BNO055.h>

// ==========================
// WIFI (STA MODE)
// ==========================
const char* WIFI_SSID = "OpenWrt2.4";
const char* WIFI_PASS = "narrowboat564";

// ==========================
// NETWORK
// ==========================
WiFiUDP udpCmd;     // for incoming commands (manual + heading)
WiFiUDP udpTelem;   // for outgoing telemetry

#define CMD_PORT    4210
#define TELEM_PORT  4211

IPAddress telemBroadcast(255, 255, 255, 255);  // broadcast telemetry

// ==========================
// MOTOR PINS
// ==========================
#define A_IN1 19
#define A_IN2 18
#define A_ENA 17

#define B_IN1 25
#define B_IN2 26
#define B_ENB 27

int dutyA = 0, dirA = 0;
int dutyB = 0, dirB = 0;

unsigned long lastCmdTime = 0;
const unsigned long FAILSAFE_TIMEOUT = 300;  // ms

// ==========================
// GPS
// ==========================
TinyGPSPlus gps;
HardwareSerial GPSserial(2);  // UART2 on pins 4/15

double gpsLat = 0, gpsLon = 0;
bool gpsFix = false;

// ==========================
// IMU / HEADING
// ==========================
Adafruit_BNO055 bno = Adafruit_BNO055(55, 0x28);

float Kp = 2.0, Ki = 0.0, Kd = 0.5;
float pidIntegral = 0, lastError = 0;

bool headingMode = false;
float targetHeading = 0;
unsigned long headingEndTime = 0;

// ==========================
// MOTOR CONTROL
// ==========================
void setMotor(int in1, int in2, int ena, int speed, int dir) {
  speed = constrain(speed, 0, 255);

  if (dir == 1) { digitalWrite(in1, HIGH); digitalWrite(in2, LOW); }
  else if (dir == -1) { digitalWrite(in1, LOW); digitalWrite(in2, HIGH); }
  else { digitalWrite(in1, LOW); digitalWrite(in2, LOW); }

  ledcWrite(ena, speed);
}

void updateMotors() {
  setMotor(A_IN1, A_IN2, 0, dutyA, dirA);
  setMotor(B_IN1, B_IN2, 1, dutyB, dirB);
}

// ==========================
// HEADING HELPERS
// ==========================
float headingError(float target, float current) {
  float e = target - current;
  while (e > 180) e -= 360;
  while (e < -180) e += 360;
  return e;
}

// ==========================
// HEADING PID WITH FULL-POWER MODE
// ==========================
void driveHeadingPID() {
  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  float error = headingError(targetHeading, heading);

  pidIntegral += error;
  float derivative = error - lastError;
  lastError = error;

  float correction = Kp * error + Ki * pidIntegral + Kd * derivative;

  int left, right;

  if (fabs(error) > 30) {
    int baseSpeed = 150;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  } else if (fabs(error) > 5) {
    int baseSpeed = 200;
    left  = constrain(baseSpeed - correction, -255, 255);
    right = constrain(baseSpeed + correction, -255, 255);
  } else {
    int full = 255;
    int trim = constrain((int)correction, -80, 80);
    left  = full - trim;
    right = full + trim;
    left  = constrain(left, 0, 255);
    right = constrain(right, 0, 255);
  }

  dutyA = abs(left);
  dirA  = (left >= 0) ? 1 : -1;

  dutyB = abs(right);
  dirB  = (right >= 0) ? 1 : -1;

  updateMotors();
}

// ==========================
// TELEMETRY (UDP)
// ==========================
unsigned long lastTelemTime = 0;
const unsigned long TELEM_INTERVAL = 50;  // ms

void sendTelemetry() {
  unsigned long now = millis();
  if (now - lastTelemTime < TELEM_INTERVAL) return;
  lastTelemTime = now;

  sensors_event_t event;
  bno.getEvent(&event, Adafruit_BNO055::VECTOR_EULER);
  float heading = event.orientation.x;

  // GPS update
  while (GPSserial.available()) gps.encode(GPSserial.read());
  gpsFix = gps.location.isValid();
  if (gpsFix) {
    gpsLat = gps.location.lat();
    gpsLon = gps.location.lng();
  }

  // Format: heading,lat,lon,fix,active
  char buf[128];
  snprintf(buf, sizeof(buf), "%.2f,%.6f,%.6f,%d,%d\n",
           heading,
           gpsLat,
           gpsLon,
           gpsFix ? 1 : 0,
           headingMode ? 1 : 0);

  udpTelem.beginPacket(telemBroadcast, TELEM_PORT);
  udpTelem.write((uint8_t*)buf, strlen(buf));
  udpTelem.endPacket();
}

// ==========================
// SETUP
// ==========================
void setup() {
  Serial.begin(115200);

  pinMode(A_IN1, OUTPUT);
  pinMode(A_IN2, OUTPUT);
  pinMode(B_IN1, OUTPUT);
  pinMode(B_IN2, OUTPUT);

  // PWM channels
  ledcSetup(0, 20000, 8);
  ledcAttachPin(A_ENA, 0);
  ledcSetup(1, 20000, 8);
  ledcAttachPin(B_ENB, 1);

  // GPS
  GPSserial.begin(9600, SERIAL_8N1, 4, 15);

  // IMU
  Wire.begin(21, 22);
  delay(1000);
  bno.begin();
  delay(1000);
  bno.setExtCrystalUse(true);
  bno.setMode(OPERATION_MODE_NDOF);

  // WiFi STA
  WiFi.mode(WIFI_STA);
  WiFi.begin(WIFI_SSID, WIFI_PASS);
  Serial.print("Connecting to WiFi");
  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
  }
  Serial.println();
  Serial.print("Connected, IP: ");
  Serial.println(WiFi.localIP());

  udpCmd.begin(CMD_PORT);
  udpTelem.begin(TELEM_PORT);

  lastCmdTime = millis();
}

// ==========================
// LOOP
// ==========================
void loop() {
  // Telemetry
  sendTelemetry();

  // Heading mode
  if (headingMode) {
    if (millis() < headingEndTime) {
      driveHeadingPID();
    } else {
      headingMode = false;
      dutyA = dutyB = 0;
      dirA = dirB = 0;
      updateMotors();
    }
  }

  // Commands
  int packetSize = udpCmd.parsePacket();
  if (packetSize) {
    char buf[64];
    int len = udpCmd.read(buf, sizeof(buf) - 1);
    buf[len] = '\0';

    // Heading command: HEAD,deg,dur_ms
    if (strncmp(buf, "HEAD", 4) == 0) {
      float deg;
      int dur;
      if (sscanf(buf, "HEAD,%f,%d", &deg, &dur) == 2) {
        headingMode = true;
        targetHeading = deg;
        headingEndTime = millis() + dur;
        pidIntegral = 0;
        lastError = 0;
      }
    } else {
      // Manual: speedA,dirA,speedB,dirB
      int sA, dA, sB, dB;
      if (sscanf(buf, "%d,%d,%d,%d", &sA, &dA, &sB, &dB) == 4) {
        if (!headingMode) {
          dutyA = sA; dirA = dA;
          dutyB = sB; dirB = dB;
          updateMotors();
        }
      }
    }

    lastCmdTime = millis();
  }

  // Failsafe
  if (!headingMode && millis() - lastCmdTime > FAILSAFE_TIMEOUT) {
    dutyA = dutyB = 0;
    dirA = dirB = 0;
    updateMotors();
  }
}


In [ ]:
import pygame
import socket
import time
import math
import os

# ==========================
# NETWORK
# ==========================
ROVER_IP = "192.168.1.50"   # adjust if needed (ESP32 STA IP)
CMD_PORT = 4210
TELEM_PORT = 4211

sock_cmd = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_telem = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_telem.bind(("", TELEM_PORT))
sock_telem.setblocking(False)

def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    sock_cmd.sendto(msg.encode(), (ROVER_IP, CMD_PORT))

def send_manual(speedA, dirA, speedB, dirB):
    msg = f"{speedA},{dirA},{speedB},{dirB}"
    sock_cmd.sendto(msg.encode(), (ROVER_IP, CMD_PORT))

def send_stop():
    send_manual(0, 0, 0, 0)

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()
pygame.joystick.init()

clock = pygame.time.Clock()
font = pygame.font.Font(None, 24)
bigfont = pygame.font.Font(None, 32)

# NES overlay
nes_img = pygame.image.load("nes.png")
nes_w, nes_h = nes_img.get_size()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover Control + NES")

# NES joystick
joysticks = {}
for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Input fields
heading_box = pygame.Rect(20, HEIGHT - 80, 150, 35)
duration_box = pygame.Rect(190, HEIGHT - 80, 150, 35)
heading_text = ""
duration_text = ""
active_heading = False
active_duration = False

heading_up = pygame.Rect(heading_box.right + 5, heading_box.y, 25, 17)
heading_down = pygame.Rect(heading_box.right + 5, heading_box.y + 18, 25, 17)
duration_up = pygame.Rect(duration_box.right + 5, duration_box.y, 25, 17)
duration_down = pygame.Rect(duration_box.right + 5, duration_box.y + 18, 25, 17)

send_button = pygame.Rect(360, HEIGHT - 80, 80, 35)
stop_button = pygame.Rect(460, HEIGHT - 80, 80, 35)

command_heading = 0.0
compass_dragging = False

# Telemetry
heading_actual = None
gps_lat = None
gps_lon = None
gps_fix = False
robot_active = False
last_telem_time = 0.0

# Cursor
cursor_visible = True
cursor_timer = 0
CURSOR_BLINK_MS = 500

def angle_wrap_deg(a):
    while a < 0:
        a += 360
    while a >= 360:
        a -= 360
    return a

def draw_compass():
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        rect = surf.get_rect(center=(x, y))
        screen.blit(surf, rect)

    if heading_actual is not None:
        rad_act = math.radians(heading_actual - 90)
        x_act = COMPASS_CX + math.cos(rad_act) * (COMPASS_R - 35)
        y_act = COMPASS_CY + math.sin(rad_act) * (COMPASS_R - 35)
        pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x_act, y_act), 4)

    rad_cmd = math.radians(command_heading - 90)
    x_cmd = COMPASS_CX + math.cos(rad_cmd) * (COMPASS_R - 60)
    y_cmd = COMPASS_CY + math.sin(rad_cmd) * (COMPASS_R - 60)
    pygame.draw.line(screen, (0, 0, 200), (COMPASS_CX, COMPASS_CY), (x_cmd, y_cmd), 4)

    label = font.render(
        f"Actual: {heading_actual:6.2f}°" if heading_actual is not None else "Actual: ---",
        True, (0, 0, 0)
    )
    screen.blit(label, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

    label2 = font.render(f"Command: {command_heading:6.2f}°", True, (0, 0, 0))
    screen.blit(label2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 30))

def compass_angle_from_mouse(mx, my):
    dx = mx - COMPASS_CX
    dy = my - COMPASS_CY
    angle = math.degrees(math.atan2(dy, dx)) + 90
    return angle_wrap_deg(angle)

def handle_telem():
    global heading_actual, gps_lat, gps_lon, gps_fix, robot_active, last_telem_time
    try:
        data, addr = sock_telem.recvfrom(256)
    except BlockingIOError:
        return
    line = data.decode().strip()
    parts = line.split(",")
    if len(parts) != 5:
        return
    try:
        heading_actual = float(parts[0])
        gps_lat = float(parts[1])
        gps_lon = float(parts[2])
        gps_fix = (parts[3] == "1")
        robot_active = (parts[4] == "1")
        last_telem_time = time.time()
    except:
        pass

running = True

while running:
    dt = clock.tick(60)
    screen.fill((240, 240, 240))

    cursor_timer += dt
    if cursor_timer >= CURSOR_BLINK_MS:
        cursor_timer = 0
        cursor_visible = not cursor_visible

    handle_telem()

    # NES overlay
    screen.blit(nes_img, (0, 0))

    # NES control
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    for joystick in joysticks.values():
        axis_0_val = joystick.get_axis(0)
        axis_4_val = joystick.get_axis(4)
        threshold = 0.9

        UP    = (axis_4_val <= -threshold)
        DOWN  = (axis_4_val >= threshold)
        LEFT  = (axis_0_val <= -threshold)
        RIGHT = (axis_0_val >= threshold)

        if LEFT:  pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)
        if RIGHT: pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)
        if UP:    pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)
        if DOWN:  pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)

        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1
        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1
        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1
        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1
        else:
            L_speed, L_dir = 0, 0
            R_speed, R_dir = 0, 0

        if UP or DOWN:
            if RIGHT:
                R_speed = int(R_speed * 0.5)
            if LEFT:
                L_speed = int(L_speed * 0.5)

        R_speed = max(0, min(255, R_speed))
        L_speed = max(0, min(255, L_speed))

    send_manual(R_speed, R_dir, L_speed, L_dir)

    # Compass + basic telemetry
    draw_compass()

    telem_lines = []
    if gps_lat is not None and gps_lon is not None:
        telem_lines.append(f"GPS: {gps_lat:.6f}, {gps_lon:.6f}")
    else:
        telem_lines.append("GPS: ---")
    telem_lines.append(f"GPS fix: {'YES' if gps_fix else 'NO'}")
    age = time.time() - last_telem_time if last_telem_time > 0 else 999
    telem_lines.append(f"Last telemetry: {age:.2f}s ago")
    telem_lines.append(f"Robot: {'ACTIVE' if robot_active else 'IDLE'}")

    y = 20
    for line in telem_lines:
        surf = font.render(line, True, (0, 0, 0))
        screen.blit(surf, (nes_w + 20, y))
        y += 22

    # Input labels
    heading_label = font.render("Heading (deg):", True, (0, 0, 0))
    duration_label = font.render("Duration (sec):", True, (0, 0, 0))
    screen.blit(heading_label, (heading_box.x, heading_box.y - 20))
    screen.blit(duration_label, (duration_box.x, duration_box.y - 20))

    # Input boxes
    pygame.draw.rect(screen, (255, 255, 255), heading_box)
    pygame.draw.rect(screen, (0, 0, 0), heading_box, 2)
    pygame.draw.rect(screen, (255, 255, 255), duration_box)
    pygame.draw.rect(screen, (0, 0, 0), duration_box, 2)

    heading_surf = font.render(heading_text, True, (0, 0, 0))
    duration_surf = font.render(duration_text, True, (0, 0, 0))
    screen.blit(heading_surf, (heading_box.x + 5, heading_box.y + 8))
    screen.blit(duration_surf, (duration_box.x + 5, duration_box.y + 8))

    if active_heading:
        cx = heading_box.x + 5 + heading_surf.get_width() + 2
        cy = heading_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + heading_box.height - 10), 1)
    elif active_duration:
        cx = duration_box.x + 5 + duration_surf.get_width() + 2
        cy = duration_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + duration_box.height - 10), 1)

    # Arrow buttons
    pygame.draw.rect(screen, (220, 220, 220), heading_up)
    pygame.draw.rect(screen, (0, 0, 0), heading_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), heading_down)
    pygame.draw.rect(screen, (0, 0, 0), heading_down, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_up)
    pygame.draw.rect(screen, (0, 0, 0), duration_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_down)
    pygame.draw.rect(screen, (0, 0, 0), duration_down, 1)

    up_surf = font.render("▲", True, (0, 0, 0))
    down_surf = font.render("▼", True, (0, 0, 0))
    screen.blit(up_surf, up_surf.get_rect(center=heading_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=heading_down.center))
    screen.blit(up_surf, up_surf.get_rect(center=duration_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=duration_down.center))

    # Buttons
    pygame.draw.rect(screen, (200, 200, 200), send_button)
    pygame.draw.rect(screen, (0, 0, 0), send_button, 2)
    screen.blit(font.render("SEND", True, (0, 0, 0)),
                (send_button.x + 15, send_button.y + 8))

    pygame.draw.rect(screen, (255, 180, 180), stop_button)
    pygame.draw.rect(screen, (0, 0, 0), stop_button, 2)
    screen.blit(font.render("STOP", True, (0, 0, 0)),
                (stop_button.x + 15, stop_button.y + 8))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy
            print("Joystick connected:", joy.get_name())

        elif event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                del joysticks[event.instance_id]
                print("Joystick removed")

        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = event.pos

            if heading_box.collidepoint(event.pos):
                active_heading = True
                active_duration = False
            elif duration_box.collidepoint(event.pos):
                active_heading = False
                active_duration = True
            else:
                active_heading = active_duration = False

            if heading_up.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val += 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)

            if heading_down.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val -= 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)

            if duration_up.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val += 0.5
                duration_text = f"{val:.1f}"

            if duration_down.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val = max(0.0, val - 0.5)
                duration_text = f"{val:.1f}"

            if send_button.collidepoint(event.pos):
                try:
                    deg = float(heading_text)
                    sec = float(duration_text)
                    send_heading(deg, int(sec * 1000))
                    command_heading = angle_wrap_deg(deg)
                    print(f"[CMD] HEAD {deg} for {sec}s")
                except:
                    print("[ERR] Invalid heading/duration")

            if stop_button.collidepoint(event.pos):
                send_stop()
                print("[CMD] STOP")

            dist = math.hypot(mx - COMPASS_CX, my - COMPASS_CY)
            if dist <= COMPASS_R:
                compass_dragging = True
                command_heading = compass_angle_from_mouse(mx, my)
                heading_text = f"{command_heading:.1f}"

        if event.type == pygame.MOUSEBUTTONUP:
            compass_dragging = False

        if event.type == pygame.MOUSEMOTION and compass_dragging:
            mx, my = event.pos
            command_heading = compass_angle_from_mouse(mx, my)
            heading_text = f"{command_heading:.1f}"

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if active_heading:
                if event.key == pygame.K_BACKSPACE:
                    heading_text = heading_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    heading_text += event.unicode

            elif active_duration:
                if event.key == pygame.K_BACKSPACE:
                    duration_text = duration_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    duration_text += event.unicode

    pygame.display.flip()

send_stop()
pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad
Joystick connected: USB Gamepad
[CMD] HEAD 1.0 for 0.5s
[CMD] HEAD 1.0 for 0.5s
[CMD] HEAD 1.0 for 0.5s
[CMD] HEAD 1.0 for 0.5s
[CMD] HEAD 1.0 for 0.5s


In [1]:
import pygame
import socket
import time
import math
import os
import csv
from datetime import datetime

# ==========================
# NETWORK
# ==========================
CMD_PORT = 4210
TELEM_PORT = 4211

sock_cmd = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_cmd.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)

sock_telem = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_telem.bind(("", TELEM_PORT))
sock_telem.setblocking(False)

CMD_ADDR = ("255.255.255.255", CMD_PORT)

def send_raw(msg: str):
    sock_cmd.sendto(msg.encode(), CMD_ADDR)

def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    send_raw(msg)

def send_manual(speedA, dirA, speedB, dirB):
    msg = f"{speedA},{dirA},{speedB},{dirB}"
    send_raw(msg)

def send_stop():
    send_manual(0, 0, 0, 0)

# Send HELLO so ESP32 learns our IP
for _ in range(5):
    send_raw("HELLO_PC")
    time.sleep(0.05)

# ==========================
# LOGGING
# ==========================
logs_dir = "logs"
os.makedirs(logs_dir, exist_ok=True)

log_file = None
log_writer = None
logging_active = False

def start_logging():
    global log_file, log_writer, logging_active
    if logging_active:
        return
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path = os.path.join(logs_dir, f"log_{ts}.csv")
    log_file = open(path, "w", newline="")
    log_writer = csv.writer(log_file)
    log_writer.writerow(["timestamp", "heading", "lat", "lon", "fix", "active"])
    logging_active = True
    print(f"[LOG] Started {path}")

def stop_logging():
    global log_file, log_writer, logging_active
    if logging_active and log_file:
        log_file.close()
        print("[LOG] Stopped")
    log_file = None
    log_writer = None
    logging_active = False

def log_row(heading, lat, lon, fix, active):
    if not logging_active or log_writer is None:
        return
    ts = datetime.now().isoformat()
    log_writer.writerow([ts, heading, lat, lon, int(fix), int(active)])

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()
pygame.joystick.init()

clock = pygame.time.Clock()
font = pygame.font.Font(None, 24)
bigfont = pygame.font.Font(None, 32)

nes_img = pygame.image.load("nes.png")
nes_w, nes_h = nes_img.get_size()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover Control + NES")

joysticks = {}
for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Input fields
heading_box = pygame.Rect(20, HEIGHT - 80, 150, 35)
duration_box = pygame.Rect(190, HEIGHT - 80, 150, 35)
heading_text = ""
duration_text = ""
active_heading = False
active_duration = False

heading_up = pygame.Rect(heading_box.right + 5, heading_box.y, 25, 17)
heading_down = pygame.Rect(heading_box.right + 5, heading_box.y + 18, 25, 17)
duration_up = pygame.Rect(duration_box.right + 5, duration_box.y, 25, 17)
duration_down = pygame.Rect(duration_box.right + 5, duration_box.y + 18, 25, 17)

send_button = pygame.Rect(360, HEIGHT - 80, 80, 35)
stop_button = pygame.Rect(460, HEIGHT - 80, 80, 35)

command_heading = 0.0
compass_dragging = False

# Telemetry
heading_actual = None
gps_lat = None
gps_lon = None
gps_fix = False
robot_active = False
last_telem_time = 0.0

# Cursor
cursor_visible = True
cursor_timer = 0
CURSOR_BLINK_MS = 500

def angle_wrap_deg(a):
    while a < 0:
        a += 360
    while a >= 360:
        a -= 360
    return a

def draw_compass():
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        rect = surf.get_rect(center=(x, y))
        screen.blit(surf, rect)

    if heading_actual is not None:
        rad_act = math.radians(heading_actual - 90)
        x_act = COMPASS_CX + math.cos(rad_act) * (COMPASS_R - 35)
        y_act = COMPASS_CY + math.sin(rad_act) * (COMPASS_R - 35)
        pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x_act, y_act), 4)

    rad_cmd = math.radians(command_heading - 90)
    x_cmd = COMPASS_CX + math.cos(rad_cmd) * (COMPASS_R - 60)
    y_cmd = COMPASS_CY + math.sin(rad_cmd) * (COMPASS_R - 60)
    pygame.draw.line(screen, (0, 0, 200), (COMPASS_CX, COMPASS_CY), (x_cmd, y_cmd), 4)

    label = font.render(
        f"Actual: {heading_actual:6.2f}°" if heading_actual is not None else "Actual: ---",
        True, (0, 0, 0)
    )
    screen.blit(label, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

    label2 = font.render(f"Command: {command_heading:6.2f}°", True, (0, 0, 0))
    screen.blit(label2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 30))

def compass_angle_from_mouse(mx, my):
    dx = mx - COMPASS_CX
    dy = my - COMPASS_CY
    angle = math.degrees(math.atan2(dy, dx)) + 90
    return angle_wrap_deg(angle)

def handle_telem():
    global heading_actual, gps_lat, gps_lon, gps_fix, robot_active, last_telem_time
    try:
        data, addr = sock_telem.recvfrom(256)
    except BlockingIOError:
        return
    line = data.decode().strip()
    parts = line.split(",")
    if len(parts) != 5:
        return
    try:
        heading_actual = float(parts[0])
        gps_lat = float(parts[1])
        gps_lon = float(parts[2])
        gps_fix = (parts[3] == "1")
        robot_active = (parts[4] == "1")
        last_telem_time = time.time()
        # logging only when active
        if robot_active:
            log_row(heading_actual, gps_lat, gps_lon, gps_fix, robot_active)
    except:
        pass

running = True

while running:
    dt = clock.tick(60)
    screen.fill((240, 240, 240))

    cursor_timer += dt
    if cursor_timer >= CURSOR_BLINK_MS:
        cursor_timer = 0
        cursor_visible = not cursor_visible

    handle_telem()

    # Start/stop logging based on active flag
    if robot_active and not logging_active:
        start_logging()
    if not robot_active and logging_active:
        stop_logging()

    # NES overlay
    screen.blit(nes_img, (0, 0))

    # NES control
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    for joystick in joysticks.values():
        axis_0_val = joystick.get_axis(0)
        axis_4_val = joystick.get_axis(4)
        threshold = 0.9

        UP    = (axis_4_val <= -threshold)
        DOWN  = (axis_4_val >= threshold)
        LEFT  = (axis_0_val <= -threshold)
        RIGHT = (axis_0_val >= threshold)

        if LEFT:  pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)
        if RIGHT: pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)
        if UP:    pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)
        if DOWN:  pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)

        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1
        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1
        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1
        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1
        else:
            L_speed, L_dir = 0, 0
            R_speed, R_dir = 0, 0

        if UP or DOWN:
            if RIGHT:
                R_speed = int(R_speed * 0.5)
            if LEFT:
                L_speed = int(L_speed * 0.5)

        R_speed = max(0, min(255, R_speed))
        L_speed = max(0, min(255, L_speed))

    send_manual(R_speed, L_dir, R_speed, R_dir)  # NOTE: adjust mapping if needed

    # Compass + telemetry
    draw_compass()

    telem_lines = []
    if gps_lat is not None and gps_lon is not None:
        telem_lines.append(f"GPS: {gps_lat:.6f}, {gps_lon:.6f}")
    else:
        telem_lines.append("GPS: ---")
    telem_lines.append(f"GPS fix: {'YES' if gps_fix else 'NO'}")
    age = time.time() - last_telem_time if last_telem_time > 0 else 999
    telem_lines.append(f"Time since last comm: {age:.2f}s")
    telem_lines.append(f"Robot: {'ACTIVE' if robot_active else 'IDLE'}")
    telem_lines.append(f"Logging: {'ON' if logging_active else 'OFF'}")

    y = 20
    for line in telem_lines:
        surf = font.render(line, True, (0, 0, 0))
        screen.blit(surf, (nes_w + 20, y))
        y += 22

    # Input labels
    heading_label = font.render("Heading (deg):", True, (0, 0, 0))
    duration_label = font.render("Duration (sec):", True, (0, 0, 0))
    screen.blit(heading_label, (heading_box.x, heading_box.y - 20))
    screen.blit(duration_label, (duration_box.x, duration_box.y - 20))

    # Input boxes
    pygame.draw.rect(screen, (255, 255, 255), heading_box)
    pygame.draw.rect(screen, (0, 0, 0), heading_box, 2)
    pygame.draw.rect(screen, (255, 255, 255), duration_box)
    pygame.draw.rect(screen, (0, 0, 0), duration_box, 2)

    heading_surf = font.render(heading_text, True, (0, 0, 0))
    duration_surf = font.render(duration_text, True, (0, 0, 0))
    screen.blit(heading_surf, (heading_box.x + 5, heading_box.y + 8))
    screen.blit(duration_surf, (duration_box.x + 5, duration_box.y + 8))

    if active_heading:
        cx = heading_box.x + 5 + heading_surf.get_width() + 2
        cy = heading_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + heading_box.height - 10), 1)
    elif active_duration:
        cx = duration_box.x + 5 + duration_surf.get_width() + 2
        cy = duration_box.y + 5
        if cursor_visible:
            pygame.draw.line(screen, (0, 0, 0), (cx, cy), (cx, cy + duration_box.height - 10), 1)

    # Arrow buttons
    pygame.draw.rect(screen, (220, 220, 220), heading_up)
    pygame.draw.rect(screen, (0, 0, 0), heading_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), heading_down)
    pygame.draw.rect(screen, (0, 0, 0), heading_down, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_up)
    pygame.draw.rect(screen, (0, 0, 0), duration_up, 1)
    pygame.draw.rect(screen, (220, 220, 220), duration_down)
    pygame.draw.rect(screen, (0, 0, 0), duration_down, 1)

    up_surf = font.render("▲", True, (0, 0, 0))
    down_surf = font.render("▼", True, (0, 0, 0))
    screen.blit(up_surf, up_surf.get_rect(center=heading_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=heading_down.center))
    screen.blit(up_surf, up_surf.get_rect(center=duration_up.center))
    screen.blit(down_surf, down_surf.get_rect(center=duration_down.center))

    # Buttons
    pygame.draw.rect(screen, (200, 200, 200), send_button)
    pygame.draw.rect(screen, (0, 0, 0), send_button, 2)
    screen.blit(font.render("SEND", True, (0, 0, 0)),
                (send_button.x + 15, send_button.y + 8))

    pygame.draw.rect(screen, (255, 180, 180), stop_button)
    pygame.draw.rect(screen, (0, 0, 0), stop_button, 2)
    screen.blit(font.render("STOP", True, (0, 0, 0)),
                (stop_button.x + 15, stop_button.y + 8))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy
            print("Joystick connected:", joy.get_name())

        elif event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                del joysticks[event.instance_id]
                print("Joystick removed")

        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = event.pos

            if heading_box.collidepoint(event.pos):
                active_heading = True
                active_duration = False
            elif duration_box.collidepoint(event.pos):
                active_heading = False
                active_duration = True
            else:
                active_heading = active_duration = False

            if heading_up.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val += 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)

            if heading_down.collidepoint(event.pos):
                try:
                    val = float(heading_text) if heading_text else 0.0
                except:
                    val = 0.0
                val -= 1.0
                heading_text = f"{val:.1f}"
                command_heading = angle_wrap_deg(val)

            if duration_up.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val += 0.5
                duration_text = f"{val:.1f}"

            if duration_down.collidepoint(event.pos):
                try:
                    val = float(duration_text) if duration_text else 0.0
                except:
                    val = 0.0
                val = max(0.0, val - 0.5)
                duration_text = f"{val:.1f}"

            if send_button.collidepoint(event.pos):
                try:
                    deg = float(heading_text)
                    sec = float(duration_text)
                    send_heading(deg, int(sec * 1000))
                    command_heading = angle_wrap_deg(deg)
                    print(f"[CMD] HEAD {deg} for {sec}s")
                except:
                    print("[ERR] Invalid heading/duration")

            if stop_button.collidepoint(event.pos):
                send_stop()
                print("[CMD] STOP")

            dist = math.hypot(mx - COMPASS_CX, my - COMPASS_CY)
            if dist <= COMPASS_R:
                compass_dragging = True
                command_heading = compass_angle_from_mouse(mx, my)
                heading_text = f"{command_heading:.1f}"

        if event.type == pygame.MOUSEBUTTONUP:
            compass_dragging = False

        if event.type == pygame.MOUSEMOTION and compass_dragging:
            mx, my = event.pos
            command_heading = compass_angle_from_mouse(mx, my)
            heading_text = f"{command_heading:.1f}"

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if active_heading:
                if event.key == pygame.K_BACKSPACE:
                    heading_text = heading_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    heading_text += event.unicode

            elif active_duration:
                if event.key == pygame.K_BACKSPACE:
                    duration_text = duration_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    duration_text += event.unicode

    pygame.display.flip()

send_stop()
stop_logging()
pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad
Joystick connected: USB Gamepad


In [ ]:
import pygame
import socket
import math
import time

# ==========================
# NETWORK
# ==========================
CMD_PORT = 4210
TELEM_PORT = 4211

sock_cmd = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_cmd.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)

sock_telem = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_telem.bind(("", TELEM_PORT))
sock_telem.setblocking(False)

CMD_ADDR = ("255.255.255.255", CMD_PORT)

def send_raw(msg):
    sock_cmd.sendto(msg.encode(), CMD_ADDR)

def send_manual(L_speed, L_dir, R_speed, R_dir):
    msg = f"{L_speed},{L_dir},{R_speed},{R_dir}"
    send_raw(msg)

# Send HELLO so ESP32 learns our IP
for _ in range(5):
    send_raw("HELLO_PC")
    time.sleep(0.05)

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()
pygame.joystick.init()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("NES Rover Test")

font = pygame.font.Font(None, 28)
bigfont = pygame.font.Font(None, 36)

clock = pygame.time.Clock()

# NES image
nes_img = pygame.image.load("nes.png")
nes_w, nes_h = nes_img.get_size()

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Telemetry
heading_actual = None
heading_error = 0
reorient_dir = 0
last_telem_time = 0

# Joystick detection
joysticks = {}
for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# ==========================
# TELEMETRY HANDLER
# ==========================
def handle_telem():
    global heading_actual, heading_error, reorient_dir, last_telem_time
    try:
        data, addr = sock_telem.recvfrom(256)
    except BlockingIOError:
        return

    parts = data.decode().strip().split(",")
    if len(parts) < 7:
        return

    try:
        heading_actual = float(parts[0])
        heading_error  = float(parts[1])
        reorient_dir   = int(parts[6])
        last_telem_time = time.time()
    except:
        pass

# ==========================
# DRAW COMPASS
# ==========================
def draw_compass():
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    # Tick marks
    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    # Labels
    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        screen.blit(surf, surf.get_rect(center=(x, y)))

    # Actual heading arrow
    if heading_actual is not None:
        rad = math.radians(heading_actual - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 40)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 40)
        pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x, y), 4)

    # Status text
    if heading_actual is not None:
        txt = font.render(f"Heading: {heading_actual:.1f}°", True, (0, 0, 0))
        screen.blit(txt, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

        txt2 = font.render(f"Error: {heading_error:.1f}°", True, (0, 0, 0))
        screen.blit(txt2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 35))

        if reorient_dir != 0:
            txt3 = font.render(
                "REORIENT: " + ("RIGHT" if reorient_dir > 0 else "LEFT"),
                True, (200, 0, 0)
            )
            screen.blit(txt3, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 60))

# ==========================
# MAIN LOOP
# ==========================
running = True

while running:
    dt = clock.tick(60)
    screen.fill((240, 240, 240))

    handle_telem()

    # NES overlay
    screen.blit(nes_img, (0, 0))

    # NES control
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    for joy in joysticks.values():
        axis_0 = joy.get_axis(0)
        axis_4 = joy.get_axis(4)
        threshold = 0.9

        UP    = axis_4 <= -threshold
        DOWN  = axis_4 >= threshold
        LEFT  = axis_0 <= -threshold
        RIGHT = axis_0 >= threshold

        # Draw red dots
        if LEFT:  pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)
        if RIGHT: pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)
        if UP:    pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)
        if DOWN:  pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)

        # Movement logic
        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1
        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1
        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1
        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1
        else:
            L_speed = R_speed = 0
            L_dir = R_dir = 0

    # Send manual command every frame
    send_manual(L_speed, L_dir, R_speed, R_dir)

    # Draw compass
    draw_compass()

    pygame.display.flip()

pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad


In [ ]:
import pygame
import socket
import time
import math

# ==========================
# NETWORK
# ==========================
CMD_PORT = 4210
TELEM_PORT = 4211

sock_cmd = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_cmd.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)

sock_telem = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock_telem.bind(("", TELEM_PORT))
sock_telem.setblocking(False)

CMD_ADDR = ("255.255.255.255", CMD_PORT)

def send_raw(msg):
    sock_cmd.sendto(msg.encode(), CMD_ADDR)

def send_manual(L_speed, L_dir, R_speed, R_dir):
    msg = f"{L_speed},{L_dir},{R_speed},{R_dir}"
    send_raw(msg)

# Send HELLO so ESP32 learns our IP
for _ in range(5):
    send_raw("HELLO_PC")
    time.sleep(0.05)

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()
pygame.joystick.init()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover NES Controller")

font = pygame.font.Font(None, 28)
bigfont = pygame.font.Font(None, 36)
clock = pygame.time.Clock()

# NES image
nes_img = pygame.image.load("nes.png")
nes_w, nes_h = nes_img.get_size()

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Telemetry
heading_actual = None
heading_error = 0
gps_lat = None
gps_lon = None
gps_fix = False
heading_mode = 0
reorient_dir = 0
last_telem_time = 0

# Joystick detection
joysticks = {}
for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# ==========================
# TELEMETRY HANDLER
# ==========================
def handle_telem():
    global heading_actual, heading_error, gps_lat, gps_lon
    global gps_fix, heading_mode, reorient_dir, last_telem_time

    try:
        data, addr = sock_telem.recvfrom(256)
    except BlockingIOError:
        return

    parts = data.decode().strip().split(",")
    if len(parts) < 7:
        return

    try:
        heading_actual = float(parts[0])
        heading_error  = float(parts[1])
        gps_lat        = float(parts[2])
        gps_lon        = float(parts[3])
        gps_fix        = (parts[4] == "1")
        heading_mode   = int(parts[5])
        reorient_dir   = int(parts[6])
        last_telem_time = time.time()
    except:
        pass

# ==========================
# DRAW COMPASS
# ==========================
def draw_compass():
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    # Tick marks
    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    # Labels
    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        screen.blit(surf, surf.get_rect(center=(x, y)))

    # Actual heading arrow
    if heading_actual is not None:
        rad = math.radians(heading_actual - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 40)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 40)
        pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x, y), 4)

    # Status text
    if heading_actual is not None:
        txt = font.render(f"Heading: {heading_actual:.1f}°", True, (0, 0, 0))
        screen.blit(txt, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

        txt2 = font.render(f"Error: {heading_error:.1f}°", True, (0, 0, 0))
        screen.blit(txt2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 35))

        if reorient_dir != 0:
            txt3 = font.render(
                "REORIENT: " + ("RIGHT" if reorient_dir > 0 else "LEFT"),
                True, (200, 0, 0)
            )
            screen.blit(txt3, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 60))

# ==========================
# MAIN LOOP
# ==========================
running = True

while running:
    dt = clock.tick(60)
    screen.fill((240, 240, 240))

    handle_telem()

    # NES overlay
    screen.blit(nes_img, (0, 0))

    # NES control
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    for joy in joysticks.values():
        axis_0 = joy.get_axis(0)
        axis_4 = joy.get_axis(4)
        threshold = 0.9

        UP    = axis_4 <= -threshold
        DOWN  = axis_4 >= threshold
        LEFT  = axis_0 <= -threshold
        RIGHT = axis_0 >= threshold

        # Draw red dots
        if LEFT:  pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)
        if RIGHT: pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)
        if UP:    pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)
        if DOWN:  pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)

        # Movement logic
        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1
        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1
        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1
        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1
        else:
            L_speed = R_speed = 0
            L_dir = R_dir = 0

    # Send manual command every frame
    send_manual(L_speed, L_dir, R_speed, R_dir)

    # Draw compass
    draw_compass()

    # Telemetry panel
    telem_lines = []
    if gps_lat is not None and gps_lon is not None:
        telem_lines.append(f"GPS: {gps_lat:.6f}, {gps_lon:.6f}")
    else:
        telem_lines.append("GPS: ---")

    telem_lines.append(f"GPS fix: {'YES' if gps_fix else 'NO'}")

    age = time.time() - last_telem_time if last_telem_time > 0 else 999
    telem_lines.append(f"Last comm: {age:.2f}s")

    telem_lines.append(f"HeadingMode: {heading_mode}")
    telem_lines.append(f"ReorientDir: {reorient_dir}")

    y = 20
    for line in telem_lines:
        surf = font.render(line, True, (0, 0, 0))
        screen.blit(surf, (nes_w + 20, y))
        y += 22

    # Quit handling
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    pygame.display.flip()

pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad


In [3]:
import socket
import time
from ipywidgets import interact, IntSlider, FloatSlider, Button, HBox, VBox, Label
from IPython.display import display

ESP_IP = "192.168.1.110"   # <-- replace with your ESP32 IP
PORT = 4210

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(1.0)

# Handshake
try:
    sock.sendto(b"HELLO_PC", (ESP_IP, PORT))
except:
    pass

# Sliders
L_pwm_slider = IntSlider(description='L PWM', min=0, max=255, step=1, value=0)
L_dir_slider = IntSlider(description='L Dir', min=-1, max=1, step=1, value=1)

R_pwm_slider = IntSlider(description='R PWM', min=0, max=255, step=1, value=0)
R_dir_slider = IntSlider(description='R Dir', min=-1, max=1, step=1, value=1)

duration_slider = FloatSlider(description='Duration (s)', min=0.1, max=3.0, step=0.1, value=1.0)

pulse_button = Button(description='Pulse', button_style='success')
status_label = Label(value='Ready')

def send_command(L_pwm, L_dir, R_pwm, R_dir):
    msg = f"{L_pwm},{L_dir},{R_pwm},{R_dir}".encode()
    sock.sendto(msg, (ESP_IP, PORT))

def on_pulse_clicked(b):
    L_pwm = L_pwm_slider.value
    L_dir = L_dir_slider.value
    R_pwm = R_pwm_slider.value
    R_dir = R_dir_slider.value
    duration = duration_slider.value

    status_label.value = f"Running for {duration:.1f}s..."
    
    # Apply command
    send_command(L_pwm, L_dir, R_pwm, R_dir)
    time.sleep(duration)
    
    # Stop
    send_command(0, 0, 0, 0)
    status_label.value = "Stopped"

pulse_button.on_click(on_pulse_clicked)

ui = VBox([
    HBox([L_pwm_slider, L_dir_slider]),
    HBox([R_pwm_slider, R_dir_slider]),
    duration_slider,
    HBox([pulse_button, status_label])
])

display(ui)


In [ ]:
import socket
import time
from ipywidgets import IntSlider, FloatSlider, Button, HBox, VBox, Label
from IPython.display import display

ESP_IP = "192.168.1.110"   # <-- change if needed
PORT = 4210

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(1.0)

# Handshake
try:
    sock.sendto(b"HELLO_PC", (ESP_IP, PORT))
except:
    pass

# ==========================
# UI
# ==========================
L_pwm_slider = IntSlider(description='L PWM', min=0, max=255, value=0)
L_dir_slider = IntSlider(description='L Dir', min=-1, max=1, value=1)

R_pwm_slider = IntSlider(description='R PWM', min=0, max=255, value=0)
R_dir_slider = IntSlider(description='R Dir', min=-1, max=1, value=1)

duration_slider = FloatSlider(description='Duration (s)', min=0.1, max=3.0, step=0.1, value=1.0)

pulse_button = Button(description='Pulse', button_style='success')
stop_button = Button(description='STOP', button_style='danger')

status_label = Label(value='Ready')

# ==========================
# NETWORK
# ==========================
def send_command(L_pwm, L_dir, R_pwm, R_dir):
    msg = f"{L_pwm},{L_dir},{R_pwm},{R_dir}".encode()
    sock.sendto(msg, (ESP_IP, PORT))

# ==========================
# ACTIONS
# ==========================
def on_pulse_clicked(b):
    L_pwm = L_pwm_slider.value
    L_dir = L_dir_slider.value
    R_pwm = R_pwm_slider.value
    R_dir = R_dir_slider.value
    duration = duration_slider.value

    status_label.value = f"Running {duration:.1f}s..."

    send_command(L_pwm, L_dir, R_pwm, R_dir)
    time.sleep(duration)

    send_command(0, 0, 0, 0)
    status_label.value = "Stopped"

def on_stop_clicked(b):
    send_command(0, 0, 0, 0)
    status_label.value = "EMERGENCY STOP"

pulse_button.on_click(on_pulse_clicked)
stop_button.on_click(on_stop_clicked)

# ==========================
# LAYOUT
# ==========================
ui = VBox([
    HBox([L_pwm_slider, L_dir_slider]),
    HBox([R_pwm_slider, R_dir_slider]),
    duration_slider,
    HBox([pulse_button, stop_button]),
    status_label
])

display(ui)

for i in range(1,10):

In [2]:
help('for')

The "for" statement
*******************

The "for" statement is used to iterate over the elements of a sequence
(such as a string, tuple or list) or other iterable object:

   for_stmt ::= "for" target_list "in" starred_list ":" suite
                ["else" ":" suite]

The "starred_list" expression is evaluated once; it should yield an
*iterable* object.  An *iterator* is created for that iterable. The
first item provided by the iterator is then assigned to the target
list using the standard rules for assignments (see Assignment
statements), and the suite is executed.  This repeats for each item
provided by the iterator.  When the iterator is exhausted, the suite
in the "else" clause, if present, is executed, and the loop
terminates.

A "break" statement executed in the first suite terminates the loop
without executing the "else" clause’s suite.  A "continue" statement
executed in the first suite skips the rest of the suite and continues
with the next item, or with the "else" clause i

In [ ]:
if isint(9/3):
    print('True')

In [ ]:
if int(9/3) == 9/3:
    print("True")

In [ ]:
if itr%2 == 0

In [3]:
for i in range (1,9,3):
    print(i)

1
4
7


In [ ]:
if int(9/3) == 9/3:
    print("True")

In [5]:
import sys

sys.argv = ['','65','120','70','170']


def calcbmi(height,weight):
    bmi = (weight/(height*height))*703
    bmi = round(bmi,2)
    return bmi
    
    
def getcategory(bmi):            

    if bmi < 0:
     
        print( "Error, please enter a numeric between 0 and 1.")
    elif bmi < 18.5:
        category = "under weight"
    elif bmi < 25:
        category = "healthy weight"
    elif bmi < 30:
        category = "overweight"
    elif bmi > 30:
        category = "obesity"
        print( "Error, please enter a numeric between 0 and 1.")
    return category
    
# print(f"{name}'s bmi: {calcbmi(height,weight)}, {getcategory(calcbmi(height,weight))}")   
if (len(sys.argv)-1) %2 != 0:
    print('The wrong number of arguments provided.') 
else:
    print('Right number of arguments provided')
    #print(f"{name}'s bmi: {calcbmi(height,weight)}, {getcategory(calcbmi(height,weight))}")
    for pair in range(1, len(sys.argv)-1):
        print(int(sys.argv[pair]))
        print(int(sys.argv[pair+1]))

Right number of arguments provided
65
120
120
70
70
170


In [7]:
import sys

# If no arguments were passed (only the script name is present)
if len(sys.argv) == 1:
    # Hard‑code fallback values
    sys.argv = ["", "1", "2", "3", "4", "5"]

# Now you can safely use sys.argv[1:]
print(sys.argv)


['', '65', '120', '70', '170']


Filename: assign4-7.py
Write a program that reads any number of sales, calculates some basic statics, and print out the result(round the average 2 decimal points). The program should give an error for non-numeric inputs. The output is shown below.  You should only use for loops.  No credit for using a while loop for this assignment.
Input:
a) python C:\Users\neda\DataProgramming\M4\assign4-7.py 50 100 200 500 600 200 10
c) python C:\Users\neda\DataProgramming\M4\assign4-7.py 900 600 fifty 100
Output:
a) Min: 10
   Max: 600
   Range: 590
   Average: 237.14
c) All inputs should be numeric

In [ ]:
import sys

# If no arguments were passed (only the script name is present)
if len(sys.argv) == 1:
    # Hard‑code fallback values
    sys.argv = ["", "50", 100 200 500 600 200 10]

# Now you can safely use sys.argv[1:]
print(sys.argv)

In [9]:
import ipywidgets as widgets
from IPython.display import display, Javascript

# Input box for the numbers
input_box = widgets.Text(
    placeholder='Enter numbers like: 50 100 200 500 600 200 10',
    description='Input:',
    layout=widgets.Layout(width='600px')
)

# Output area
output = widgets.Textarea(
    value='',
    description='sys.argv:',
    layout=widgets.Layout(width='600px')
)

# Copy button
copy_button = widgets.Button(
    description='Copy',
    button_style='success',
    tooltip='Copy to clipboard'
)

# Generate button
generate_button = widgets.Button(
    description='Generate sys.argv',
    button_style='info'
)

# --- Logic ---

def generate_argv(_):
    raw = input_box.value.strip()
    if raw == "":
        argv_list = [""]  # only the blank script name
    else:
        parts = raw.split()
        argv_list = [""] + parts  # prepend blank like sys.argv[0]
    output.value = str(argv_list)

def copy_to_clipboard(_):
    js = Javascript(f"""
    navigator.clipboard.writeText(`{output.value}`);
    """)
    display(js)

generate_button.on_click(generate_argv)
copy_button.on_click(copy_to_clipboard)

# Display everything
display(input_box, generate_button, output, copy_button)


Text(value='', description='Input:', layout=Layout(width='600px'), placeholder='Enter numbers like: 50 100 200…

Button(button_style='info', description='Generate sys.argv', style=ButtonStyle())

Textarea(value='', description='sys.argv:', layout=Layout(width='600px'))

Button(button_style='success', description='Copy', style=ButtonStyle(), tooltip='Copy to clipboard')